# 01 Train DBGNN
Train/evaluate the model once, then generate key training figures explicitly.


## Setup


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from notebook_runner import run_notebook_script


## Configuration


In [ ]:
SCRIPT_NAME = '01_train_dbgnn'
overrides = {
    'dataset_key': 'hospital',
    'render_all_plots': False,
    # Fast default for notebook iteration:
    'use_node2vec_features': False,
    'train_from_scratch': False,

    # Optional full training override:
    # 'epochs': 200,
}


## Compute Shared State


In [ ]:
state = run_notebook_script(SCRIPT_NAME, overrides=overrides)
print('Run name:', state['run_name'])
print('Dataset:', state['dataset_name'])

print('Palette:', {'eventblue': state['EVENT_BLUE'], 'snapshotorange': state['SNAPSHOT_ORANGE'], 'edgegray': state['EDGE_GRAY']})

### Plot 1: Training Curves (Loss + Balanced Accuracy)


In [ ]:
train_info = state.get('train_info', {})
losses = train_info.get('losses', [])
train_ba = train_info.get('train_ba', None)
test_ba = train_info.get('test_ba', None)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
if losses:
    axes[0].plot(range(1, len(losses) + 1), losses, label='loss', color=state['EVENT_BLUE'])
axes[0].set_title('Training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

if train_ba is not None and test_ba is not None:
    axes[1].plot(range(1, len(train_ba) + 1), train_ba, label='train', color=state['EVENT_BLUE'])
    axes[1].plot(range(1, len(test_ba) + 1), test_ba, label='test', color=state['SNAPSHOT_ORANGE'])
    axes[1].legend()
axes[1].set_title('Balanced accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Balanced accuracy')

plt.tight_layout()
safe_run_name = ''.join(ch if (ch.isalnum() or ch in '-_.') else '_' for ch in str(state['run_name']))
fig.savefig(Path(state['PLOT_DIR']) / f'{safe_run_name}_training_curves.pdf', bbox_inches='tight')
plt.show()
